<hr><br>

## Full Workflow of Data Processing + Training

<hr><br>

## Data Processing

### Loading Required Libraries

Before we proceed with the experiment setup, let's first import all the necessary libraries. These libraries will be used for:
- Mesh processing and visualization (`vtk`, `pyvista`)
- Data handling and file operations (`pathlib`, `concurrent.futures`)
- Progress tracking and visualization (`tqdm`, `matplotlib`)
- PyTorch provides data primitives: `torch.utils.data.Dataset` that allow you to use pre-loaded datasets as well as your own data. `Dataset` stores the samples and their corresponding labels.
- Important utilities for data processing and training, testing DoMINO (`physicsnemo.utils.domino.utils`, `physicsnemo.utils.domino.vtk_file_utils`)

In [1]:
# importing relevant libraries

import os
import random
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from typing import Union

import numpy as np
import pyvista as pv
import vtk
from stl import mesh
from tqdm import tqdm

from physicsnemo.utils.domino.utils import *
from physicsnemo.utils.domino.vtk_file_utils import *
from torch.utils.data import Dataset

### Defining Experiment Parameters and Variables

In [2]:
# Physical variables
VOLUME_VARS = ["p", "U"]
SURFACE_VARS = ["p", "U", "T"]
GLOBAL_PARAMS_TYPES = {"air_density" : "scalar", "inlet_velocity_x" : "vector", "inlet_velocity_y" : "vector"}
GLOBAL_PARAMS_REFERENCE = {"air_density" : 1.225, "inlet_velocity_x": [0.0], "inlet_velocity_y": [-2.0]}

### Defining File Paths

In [3]:
DATA_DIR = Path("/Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset")

In [ ]:
!ls {DATA_DIR}

Train_info                  Train_vtk
Train_prepared_surface_data Train_vtp
Train_stl


In [4]:
def setup_environment(data_dir: str):
    """
    Sets up the working environment by defining the folder paths for training, validation, and test splits.
    
    This function helps organize the dataset for preprocessing and downstream training.
    
    Returns:
        dataset_paths: Dict with paths to VTP/STL files for each split.
        info_paths: Dict with paths to global parameter info files.
        stl_paths: Dict with paths to STL geometry files.
        surface_paths: Dict with paths to save processed NumPy surface data.
    """
    print("=== Environment Setup ===")
    print(f"Current data directory: {data_dir}")

    paths = ["train", "validation", "test"]
    # Paths to raw VTk/mesh data
    dataset_paths = {k: os.path.join(data_dir, f"{k}_vtk") for k in paths}

    # Paths to vtu files
    vtu_paths = {k: os.path.join(data_dir, f"{k}_vtu") for k in paths}

    # Paths to vtp/ mesh data
    vtp_paths = {k: os.path.join(data_dir, f"{k}_vtp") for k in paths}
    
    # Paths to global parameter info files (text files)
    info_paths = {k: os.path.join(data_dir, f"{k}_info") for k in paths}
    
    # Paths to STL files
    stl_paths = {k: os.path.join(data_dir, f"{k}_stl") for k in paths}
    
    # Paths to save processed surface data as NumPy arrays
    surface_paths = {k: os.path.join(data_dir, f"{k}_prepared_surface_data") for k in dataset_paths}

    # Print all paths for confirmation
    print("\nConfigured directory paths:")
    for split in paths:
        print(f"  {split.capitalize()} data: {dataset_paths[split]}")
        print(f"  {split.capitalize()} data: {vtu_paths[split]}")
        print(f"  {split.capitalize()} vtp: {vtp_paths[split]}")
        print(f"  {split.capitalize()} info: {info_paths[split]}")
        print(f"  {split.capitalize()} STL: {stl_paths[split]}")
        print(f"  {split.capitalize()} prepared surface data: {surface_paths[split]}\n")

    return dataset_paths, vtu_paths, vtp_paths, info_paths, stl_paths, surface_paths

dataset_paths, vtu_paths, vtp_paths, info_paths, stl_paths, surface_paths = setup_environment(DATA_DIR)

=== Environment Setup ===
Current data directory: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset

Configured directory paths:
  Train data: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtk
  Train data: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtu
  Train vtp: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtp
  Train info: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_info
  Train STL: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_stl
  Train prepared surface data: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_prepared_surface_data

  Validation data: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/validation_vtk
  Validation data: /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/validation_vtu
  Validation vtp: /Users/n

### Generating STL, VTP and VTU files from legacy VTK

In [6]:
# use helper file file_format_converter.py
import file_format_converter
file_format_converter.convert_vtk_to_stl_vtp_vtu_batch(dataset_paths, stl_paths, vtp_paths, vtu_paths)


=== Starting Conversion Process ===

Processing 5 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtk → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_stl...


Converting train to STL: 100%|██████████| 5/5 [00:26<00:00,  5.28s/it]



Processing 5 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtk → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtp...


Converting train to VTP: 100%|██████████| 5/5 [00:21<00:00,  4.21s/it]



Processing 5 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtk → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtu...


Converting train to VTU: 100%|██████████| 5/5 [00:43<00:00,  8.63s/it]


[WARNING] No .vtk files found in /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/validation_vtk
[WARNING] No .vtk files found in /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/test_vtk

=== All Conversions Completed Successfully ===


### Conversion from NPY, VTP and Info Files to DoMINO ready NPY format

In [7]:
# use helper file stl_vtp_to_npy.py
import generate_npy_data
for out_path in surface_paths.values():
    !rm -rf {out_path}
try:
    generate_npy_data.process_surface_data_batch(vtp_paths, vtu_paths, info_paths, stl_paths, surface_paths)
except Exception as e:
    print("[!] Process Ended Prematurely. Deleting Progress")
    for out_path in surface_paths.values():
        !rm -rf {out_path}
    raise e

=== Starting Processing ===

Processing 5 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_vtp → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/train_prepared_surface_data...


Processing train: 100%|██████████| 5/5 [00:11<00:00,  2.37s/it]



Processing 0 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/validation_vtp → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/validation_prepared_surface_data...


Processing validation: 0it [00:00, ?it/s]



Processing 0 files from /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/test_vtp → /Users/nguyentuanphong/Desktop/ASTAR Internship/PhysicsNeMo/HDB/Dataset/test_prepared_surface_data...


Processing test: 0it [00:00, ?it/s]

=== All Processing Completed Successfully ===


### Checking output files

In [9]:
import pandas as pd
import random

for split, path in surface_paths.items():
    files = list(Path(path).glob("*.npy"))

    if not files:
        print(f"\nNo processed files found for {split} split")
        continue

    rows = []
    for f in random.sample(files, min(4, len(files))):
        try:
            d = np.load(f, allow_pickle=True).item()
            row = {
                "file": f.stem,
                "velocity_x (m/s)": d["global_params_values"][1],
                "velocity_y (m/s)": d["global_params_values"][2],
                "density (kg/m³)": d["global_params_values"][0],
                "surface_points": d["surface_mesh_centers"].shape[0],
                "stl_faces": d["stl_faces"].shape[0] // 3,
                "volume_points": d["volume_mesh_centers"].shape[0],
            }

            # Surface fields
            surface_fields = d["surface_fields"]
            surface_names = ["pressure", "U_x", "U_y", "U_z"]
            for j, name in enumerate(surface_names[:surface_fields.shape[1] if surface_fields.ndim > 1 else 1]):
                col = surface_fields[:, j] if surface_fields.ndim > 1 else surface_fields
                row[f"surf_{name}_min"], row[f"surf_{name}_max"] = col.min(), col.max()

            # Volume fields
            volume_fields = d["volume_fields"]
            volume_names = ["p", "U_x", "U_y", "U_z"]
            for j, name in enumerate(volume_names[:volume_fields.shape[1] if volume_fields.ndim > 1 else 1]):
                col = volume_fields[:, j] if volume_fields.ndim > 1 else volume_fields
                row[f"vol_{name}_min"], row[f"vol_{name}_max"] = col.min(), col.max()

            rows.append(row)
        except Exception as e:
            rows.append({"file": f.name, "error": str(e)})

    print(f"\n{split.upper()} Split — {len(files)} files total:")
    display(pd.DataFrame(rows))

    # Examine the data fields available
    print(f"\nAvailable fields:")
    for name in d.keys():
        val = d[name]
        if hasattr(val, "shape"):
            print(f"  {name:24s} | shape = {str(getattr(val, 'shape', ())):12s} | range=[{np.min(val):8.4f}, {np.max(val):11.4f}]")
        else:
            print(f"  {name:24s} | value = {val}")


TRAIN Split — 5 files total:


,file,velocity_x (m/s),velocity_y (m/s),density (kg/m³),surface_points,stl_faces,volume_points,surf_pressure_min,surf_pressure_max,surf_U_x_min,...,surf_U_z_min,surf_U_z_max,vol_p_min,vol_p_max,vol_U_x_min,vol_U_x_max,vol_U_y_min,vol_U_y_max,vol_U_z_min,vol_U_z_max
0,case_HDB_1501b095_N,0.0,-2.0,1.225,267980,532074,5812410,-5180.539551,0.263134,-0.021001,...,-0.059680,1.459768,-5406.108887,0.689559,-2.452811,2.081164,-4.213025,1.788788,-2.221363,2.00000
1,case_HDB_1507b002_N,0.0,-2.0,1.225,182172,362526,2608527,-1716.756592,0.000000,-0.007814,...,-0.043331,1.562500,-1907.888184,0.000000,-1.532820,1.750000,-3.684578,1.630602,-0.606156,1.75000
2,case_HDB_1523b079_N,0.0,-2.0,1.225,249069,494620,4190474,-3434.941895,0.430515,-0.016554,...,-0.109325,1.609204,-3650.184326,0.906862,-2.475512,2.525154,-4.197865,2.008727,-1.234017,2.54399
3,case_HDB_1528b101_N,0.0,-2.0,1.225,256180,508502,2892829,-2266.798340,0.000000,-0.019768,...,-0.100576,1.000000,-2473.389648,0.000000,-1.955039,2.221251,-3.882948,1.500000,-1.131433,1.79027



Available fields:
  stl_coordinates          | shape = (254253, 3)  | range=[-3000.0000,   3000.0000]
  stl_centers              | shape = (508502, 3)  | range=[-3000.0000,   3000.0000]
  stl_faces                | shape = (1525506,)   | range=[  0.0000, 254252.0000]
  stl_areas                | shape = (508502,)    | range=[  0.1393,   5000.1006]
  surface_mesh_centers     | shape = (256180, 3)  | range=[-3000.0000,   3000.0000]
  surface_normals          | shape = (256180, 3)  | range=[ -1.0000,      1.0000]
  surface_areas            | shape = (256180,)    | range=[  0.2961,  10000.1699]
  surface_fields           | shape = (256180, 4)  | range=[-2266.7983,      1.0625]
  volume_mesh_centers      | shape = (2892829, 3) | range=[-3000.0000,   3000.0000]
  volume_fields            | shape = (2892829, 5) | range=[-2473.3896,    303.1170]
  filename                 | value = case_HDB_1528b101_N.vtp
  global_params_values     | shape = (3,)         | range=[ -2.0000,      1.2250]
  glob